# Week 3 Mini Project: Factory Production Analysis & Performance Monitoring Dashboard
**Course**: MACSE502 - Programming for Data Science Lab  
**Student Name**: VEDANT NIMKAR  
**Registration Number**: 26MML0045  
**Date**: 23-07-2026 | **Faculty**: Dr. Rajasekhara Babu M  

---
## 1. Executive Summary & Problem Formulation
Industrial manufacturing facilities track high-frequency production volume across multiple assembly lines and operating shifts. This project utilizes high-performance multi-dimensional array operations via **NumPy** to model factory throughput, calculate dispersion statistics, slice critical operational windows, and perform matrix transformations. The **Mini Project Extension** implements an advanced **Factory Performance Analytics Dashboard** that computes real-time line capabilities ($C_p$), flags shift bottlenecks, models machine thermal/mechanical strain indices, and generates actionable predictive maintenance schedules.

### Objectives
1. Construct a $4 \times 5$ NumPy production matrix representing 4 production lines across 5 shifts.
2. Perform array aggregations: overall mean, standard deviation, 2D sub-array slicing, and matrix reshaping to $(5, 4)$.
3. Conduct element-wise multiplication with a $5 \times 4$ machine efficiency coefficient matrix.
4. Implement the **Mini Project Extension**: Statistical Process Control ($C_p, UCL, LCL$), shift fatigue anomaly detection, strain matrix modeling, and visual dashboard.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('NumPy version:', np.__version__)

## 2. NumPy Array Construction & Core Matrix Operations

In [ ]:
# Define 4 Lines and 5 Shifts
lines = ['Line A (Assembly)', 'Line B (Machining)', 'Line C (Fabrication)', 'Line D (Packaging)']
shifts = ['Shift 1 (Morning)', 'Shift 2 (Afternoon)', 'Shift 3 (Evening)', 'Shift 4 (Night)', 'Shift 5 (Graveyard)']

production = np.array([
    [145, 152, 138, 120, 110],
    [168, 175, 160, 142, 125],
    [130, 135, 128, 115,  98],
    [185, 190, 178, 165, 150]
], dtype=np.int32)

print(f'Production Matrix Dimensions: {production.shape}')
display(pd.DataFrame(production, index=lines, columns=shifts))

# Step 1 & 2: Mean and Standard Deviation
mu = np.mean(production)
sigma = np.std(production)
print(f'Factory Overall Mean Output: {mu:.2f} units/shift')
print(f'Factory Standard Deviation: {sigma:.2f} units')

# Step 3: Sub-array slicing (First 3 lines, Last 2 shifts)
sliced = production[:3, -2:]
print('\nSliced Sub-Matrix ([:3, -2:]):')
display(pd.DataFrame(sliced, index=lines[:3], columns=shifts[-2:]))

# Step 4: Reshape into (5, 4)
reshaped = production.reshape(5, 4)
print(f'\nReshaped to (5, 4):\n{reshaped}')

# Step 5: Element-wise multiplication with Machine Efficiency Matrix
efficiency_5x4 = np.array([
    [0.92, 0.88, 0.95, 0.91],
    [0.89, 0.94, 0.86, 0.93],
    [0.96, 0.90, 0.92, 0.85],
    [0.87, 0.84, 0.90, 0.95],
    [0.83, 0.86, 0.81, 0.89]
])
effective_output = reshaped * efficiency_5x4
print('\nEffective Production Output Matrix (Reshaped 5x4 * Efficiency):')
print(effective_output.round(1))

## 3. Mini Project Extension: Factory Performance Analytics & Process Capability ($C_p$)

In [ ]:
efficiency_4x5 = efficiency_5x4.T[:4, :5]
effective_4x5 = production * efficiency_4x5

line_totals = np.sum(production, axis=1)
line_effective_totals = np.sum(effective_4x5, axis=1)
line_means = np.mean(production, axis=1)
line_stds = np.std(production, axis=1)

# Process Capability Cp = (USL - LSL) / (6 * sigma)
USL, LSL = 195.0, 105.0
Cp = (USL - LSL) / (6 * line_stds)

line_summary = pd.DataFrame({
    'Line': lines,
    'GrossUnits': line_totals,
    'EffectiveUnits': line_effective_totals.round(1),
    'Mean': line_means.round(1),
    'StdDev': line_stds.round(2),
    'Capability_Cp': Cp.round(2),
    'Assessment': ['Capable' if c >= 1.0 else 'High Variance' for c in Cp]
})
display(line_summary)

# Machine Strain Matrix S = P / eta^2
strain_matrix = production / (efficiency_4x5 ** 2)
risk_matrix = (strain_matrix - np.min(strain_matrix)) / (np.max(strain_matrix) - np.min(strain_matrix))
print('\nNormalized Predictive Maintenance Risk (0.0=Low, 1.0=Critical):')
display(pd.DataFrame(risk_matrix.round(2), index=lines, columns=shifts))

## 4. Visual Factory Performance Cockpit

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Industrial Manufacturing Factory Production Analytics Dashboard\nVEDANT NIMKAR (26MML0045) | Week 3 Mini Project', fontsize=15, weight='bold', y=0.98)

# Heatmap of gross production
sns.heatmap(production, annot=True, fmt='d', cmap='YlGnBu', xticklabels=[s.split(' ')[0] for s in shifts], yticklabels=[l.split(' ')[1] for l in lines], ax=axes[0, 0])
axes[0, 0].set_title('Production Line vs. Shift Output Heatmap', weight='bold')
axes[0, 0].set_xlabel('Operational Shift')
axes[0, 0].set_ylabel('Production Line')

# Gross vs Effective output
x = np.arange(len(lines))
w = 0.35
axes[0, 1].bar(x - w/2, line_totals, w, label='Gross Output (Nominal)', color='#2b5c8f', edgecolor='black')
axes[0, 1].bar(x + w/2, line_effective_totals, w, label='Effective Output (Adjusted)', color='#e27d60', edgecolor='black')
axes[0, 1].set_title('Line Output Comparison: Gross vs. Efficiency-Adjusted', weight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels([l.split(' ')[1] for l in lines])
axes[0, 1].legend()

# SPC Control Chart
axes[1, 0].errorbar(x, line_means, yerr=2*line_stds, fmt='o', color='#1d3557', ecolor='#e63946', elinewidth=2, capsize=6, markersize=8, label='Line Mean +/- 2 Sigma')
axes[1, 0].axhline(mu, color='green', linestyle='--', linewidth=1.5, label=f'Factory Mean ({mu:.1f})')
axes[1, 0].axhline(mu + 2*sigma, color='red', linestyle=':', label=f'UCL ({mu + 2*sigma:.1f})')
axes[1, 0].axhline(mu - 2*sigma, color='red', linestyle=':', label=f'LCL ({mu - 2*sigma:.1f})')
axes[1, 0].set_title('Statistical Process Control (SPC): Line Means & Control Bounds', weight='bold')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels([l.split(' ')[1] for l in lines])
axes[1, 0].legend(loc='lower left')

# Strain Risk Heatmap
sns.heatmap(risk_matrix, annot=True, fmt='.2f', cmap='Reds', xticklabels=[s.split(' ')[0] for s in shifts], yticklabels=[l.split(' ')[1] for l in lines], ax=axes[1, 1])
axes[1, 1].set_title('Predictive Maintenance Strain Risk Matrix', weight='bold')
axes[1, 1].set_xlabel('Operational Shift')
axes[1, 1].set_ylabel('Production Line')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()